# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_<MSSV>/)
# Colab: nếu chưa có repo, bỏ comment 2 dòng dưới, rồi đặt REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"
# !git clone https://github.com/VinUni-AI20k/K4-Track4-Day1-Neuralnetwork.git
# REPO_ROOT = "/content/K4-Track4-Day1-Neuralnetwork"

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"torch {torch.__version__} | device = {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx

: 

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Chạy scripts/split_data.py (bỏ qua nếu data/processed/*.npz đã có sẵn)
train_npz = f"{REPO_ROOT}/data/processed/train.npz"
eval_npz = f"{REPO_ROOT}/data/processed/eval.npz"
if not (os.path.exists(train_npz) and os.path.exists(eval_npz)):
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)
else:
    print("data/processed/*.npz đã tồn tại, bỏ qua split_data.py")

# prepare_data tự in kích thước các tập, accuracy "luôn đoán lớp đa số" trên val,
# và mean/std của 10 cột số sau chuẩn hoá (phải xấp xỉ 0 / 1)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=f"{REPO_ROOT}/data/processed")

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
import math
import matplotlib.pyplot as plt

# ----- 1. model và số tham số -----
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
assert n_params == EXPECTED_PARAMS[(256, 128)], n_params
print(f"số tham số = {n_params}  ==  EXPECTED_PARAMS[(256,128)] = {EXPECTED_PARAMS[(256, 128)]}")

# ----- 2. một lô ngẫu nhiên qua model -----
set_seed(0)
xb_rand = torch.randn(8, 54, device=device)
logits_rand = model(xb_rand)
print("logits.shape =", tuple(logits_rand.shape))
assert logits_rand.shape == (8, 7)

# ----- 3. loss bước 0 trên val, kỳ vọng xấp xỉ ln 7 -----
step0 = evaluate(model, data["X_val"], data["y_val"], loss_name="ce")
print(f"step0 val loss = {step0['loss']:.4f}   ln(7) = {math.log(7):.4f}")

# ----- 4. quá khớp 20 mẫu -----
set_seed(0)
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
overfit_opt = build_optimizer("adam", overfit_model.parameters(), lr=1e-2)
x20, y20 = data["X_tr"][:20], data["y_tr"][:20]
overfit_model.train()
overfit_losses = []
for step in range(300):
    overfit_opt.zero_grad(set_to_none=True)
    loss20 = torch.nn.functional.cross_entropy(overfit_model(x20), y20)
    loss20.backward()
    overfit_opt.step()
    overfit_losses.append(loss20.item())
print(f"quá khớp 20 mẫu: loss bước đầu = {overfit_losses[0]:.4f} -> loss bước cuối = {overfit_losses[-1]:.6f}")

plt.figure(figsize=(5, 3))
plt.plot(overfit_losses)
plt.xlabel("bước"); plt.ylabel("CE loss"); plt.title("Quá khớp 20 mẫu")
plt.tight_layout(); plt.show()

# ----- 5. grad norm từng tham số sau một lần backward -----
model.zero_grad(set_to_none=True)
xb_grad, yb_grad = data["X_tr"][:512], data["y_tr"][:512]
loss_grad = torch.nn.functional.cross_entropy(model(xb_grad), yb_grad)
loss_grad.backward()
for name, p in model.named_parameters():
    gn = p.grad.norm().item() if p.grad is not None else None
    print(f"{name:20s} grad_norm = {gn}")
    assert p.grad is not None and gn != 0.0

**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# ----- chọn lr cho baseline BẰNG VAL (không dùng eval) -----
lr_candidates = [0.01, 0.03, 0.05, 0.1]
lr_search = []
for lr in lr_candidates:
    r = run_experiment({**DEFAULT_CFG, "lr": lr, "epochs": 8, "exp_id": f"lr-search-{lr}"}, data)
    lr_search.append((lr, r["summary"]["val_macro_f1"], r["summary"]["best_val_loss"]))
    print(f"lr={lr:<6} val_macro_f1={r['summary']['val_macro_f1']:.4f}  best_val_loss={r['summary']['best_val_loss']:.4f}")

best_lr = max(lr_search, key=lambda t: t[1])[0]
print()
print("lr chọn cho baseline (val_macro_f1 cao nhất):", best_lr)

cfg = {**DEFAULT_CFG, "lr": best_lr}

# ----- baseline với >= 2 seed (ở đây dùng 3) -----
baseline_results = []
for s in (1, 2, 3):
    exp_id = f"base-s{s}"
    r = run_experiment({**cfg, "seed": s, "exp_id": exp_id}, data)
    save_result(r, f"{OUT_DIR}/results")
    plot_run(r, f"{OUT_DIR}/figures/{exp_id}.png")
    baseline_results.append(r)

base_f1 = np.array([r["summary"]["val_macro_f1"] for r in baseline_results])
base_acc = np.array([r["summary"]["val_acc"] for r in baseline_results])
base_f1_std = base_f1.std(ddof=1)
print()
print(f"val_macro_f1: {base_f1.mean():.4f} +/- {base_f1_std:.4f}  (qua {len(baseline_results)} seed)")
print(f"val_acc:      {base_acc.mean():.4f} +/- {base_acc.std(ddof=1):.4f}")
print(f"nguong nhieu 2*std(val_macro_f1) = {2 * base_f1_std:.4f}")

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm, lặp qua danh sách cfg tự thiết kế (mỗi cfg chỉ đổi MỘT yếu tố so với `cfg` baseline).
# TODO: thêm/đổi các cấu hình theo chủ đề bạn chọn (loss, optimizer, hparam, dropout, clipping, amp, init).
exp_configs = [
    {**cfg, "exp_id": "opt-adam", "group": "optimizer", "description": "Đổi optimizer sang Adam",
     "optimizer": "adam", "lr": 0.001},
]

group_results = {}
for exp_cfg in exp_configs:
    r = run_experiment(exp_cfg, data)
    save_result(r, f"{OUT_DIR}/results")
    plot_run(r, f"{OUT_DIR}/figures/{exp_cfg['exp_id']}.png")
    group_results.setdefault(exp_cfg["group"], []).append(r)

    s = r["summary"]
    d_f1 = s["val_macro_f1"] - base_f1.mean()
    beyond_noise = abs(d_f1) > 2 * base_f1_std
    nhan_xet = "Co" if beyond_noise else "Khong"
    print(f"[{exp_cfg['exp_id']}] val_acc={s['val_acc']:.4f}  val_macro_f1={s['val_macro_f1']:.4f}  "
          f"delta_f1_vs_baseline={d_f1:+.4f}  vuot_nguong_nhieu={nhan_xet}")

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# ảnh chồng theo nhóm: so sánh val_loss của baseline (đại diện) với các thí nghiệm cùng nhóm
for group_name, results_in_group in group_results.items():
    plot_compare([baseline_results[0], *results_in_group], "val_loss",
                 f"{OUT_DIR}/figures/compare_{group_name}.png",
                 title=f"So sánh val_loss - nhóm {group_name}")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# Chọn cấu hình CHỈ bằng VAL. Mặc định dưới đây là baseline (seed 1); nếu một thí nghiệm ở Part 3
# vượt ngưỡng nhiễu và tốt hơn baseline trên val_macro_f1, đổi cfg_final/result_final sang thí nghiệm đó.
cfg_final = cfg
result_final = baseline_results[0]

eval_scores_final = final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
print("EVAL (cấu hình cuối cùng):", eval_scores_final["accuracy"], eval_scores_final["macro_f1"])

# baseline (để đối chiếu trong báo cáo/bảng); không cần nộp nếu không dùng làm cấu hình cuối cùng
eval_scores_base = final_eval(cfg, baseline_results[0], data,
                              f"{OUT_DIR}/predictions_eval_baseline.csv",
                              out_json=f"{OUT_DIR}/eval_result_baseline.json")
print("EVAL (baseline):          ", eval_scores_base["accuracy"], eval_scores_base["macro_f1"])

In [ ]:
import pandas as pd

per_class = pd.DataFrame(eval_scores_final["per_class"])
print(per_class)

cm = np.array(eval_scores_final["confusion_matrix"])
print()
print("ma trận nhầm lẫn (hàng = thật, cột = dự đoán):")
print(pd.DataFrame(cm))

worst = per_class.sort_values("f1").iloc[0]
print()
print(f"lớp khó nhất: {int(worst['cls'])}  (f1={worst['f1']:.4f}, support={int(worst['support'])})")

**Phân tích lỗi (tự viết):** ...

In [ ]:
results = load_results(f"{OUT_DIR}/results")

base_exp_id = baseline_results[0]["cfg"]["exp_id"]  # seed đã chấm điểm eval ở trên, dùng làm "baseline" trong bảng
rows = []
for r in results:
    exp_id = r["cfg"]["exp_id"]
    if exp_id == cfg_final["exp_id"]:
        rows.append(to_row(r, eval_scores=eval_scores_final))
    elif exp_id == base_exp_id:
        rows.append(to_row(r, eval_scores=eval_scores_base))
    else:
        rows.append(to_row(r))

write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
print(f"đã ghi {OUT_DIR}/experiments.xlsx ({len(rows)} dòng)")
# TODO: mở {OUT_DIR}/experiments.xlsx bằng Excel/LibreOffice để công thức tính lại; sau đó:
#   - điền sheet "Seeds": exp_id của các lần chạy baseline (base-s1, base-s2, base-s3)
#   - viết nhận xét ngắn ở sheet "Summary" cho từng nhóm chủ đề bạn đã thử